In [ ]:
# Count the total number of incidents recorded in the cyber_secuitry table.
df = await query_db('''SELECT COUNT(*) AS total_incidents FROM cyber_secuitry''')

if df is not None and not df.empty:
    if len(df) == 1 and len(df.columns) == 1:
        result_value = df.iloc[0, 0]
        col_name = str(df.columns[0])
        print("=" * 50)
        print(col_name + ": " + str(result_value))
        print("=" * 50)
else:
    print("No data returned")

df

,total_incidents
0,500


In [ ]:
# This query counts the number of incidents for each project in the cyber_secuitry table.
df = await query_db('''SELECT project, COUNT(*) AS incident_count FROM cyber_secuitry GROUP BY project''')

if df is not None and not df.empty:
    print("Results: " + str(len(df)) + " rows")
else:
    print("No data returned")

df

,project,incident_count
0,ORION,115
1,ATLAS,115
2,APOLLO,134
3,MERCURY,136


In [ ]:
# Query to find the different Threat_Types and their frequency in the cyber_secuitry table.
df = await query_db('''SELECT threat_category, COUNT(*) AS frequency FROM cyber_secuitry GROUP BY threat_category ORDER BY frequency DESC''')

if df is not None and not df.empty:
    print("Results: " + str(len(df)) + " rows")
else:
    print("No data returned")

df

,threat_category,frequency
0,Data Exfiltration,93
1,Malware,92
2,Insider Threat,86
3,DDoS,78
4,Phishing,77
5,Ransomware,74


In [ ]:
# This query retrieves the distribution of Priority levels (Low, Medium, High, Critical) in the cyber_secuitry table by counting the number of incidents for each priority level and ordering the results by frequency in descending order.
df = await query_db('''SELECT priority, COUNT(*) AS frequency FROM cyber_secuitry GROUP BY priority ORDER BY frequency DESC''')

if df is not None and not df.empty:
    print("Results: " + str(len(df)) + " rows")
else:
    print("No data returned")

df

,priority,frequency
0,Low,136
1,Critical,125
2,High,121
3,Medium,118


In [ ]:
# Query to find which Sensor_Source detects the most incidents in the cyber_secuitry table.
df = await query_db('''SELECT sensor_source, COUNT(*) AS incident_count FROM cyber_secuitry GROUP BY sensor_source ORDER BY incident_count DESC LIMIT 1''')

if df is not None and not df.empty:
    print("Sensor Source with the most incidents: " + df.iloc[0]['sensor_source'] + " with " + str(df.iloc[0]['incident_count']) + " incidents")
else:
    print("No data returned")

df

""


In [ ]:
# Query to find which SOC Owner team handles the most incidents in the cyber_secuitry table.
df = await query_db('''SELECT soc_owner, COUNT(*) AS incident_count FROM cyber_secuitry GROUP BY soc_owner ORDER BY incident_count DESC LIMIT 1''')

if df is not None and not df.empty:
    print("SOC Owner team handling the most incidents: " + df.iloc[0]['soc_owner'] + " with " + str(df.iloc[0]['incident_count']) + " incidents")
else:
    print("No data returned")

df

,soc_owner,incident_count
0,SOC Team,176


In [ ]:
# Query to find which Threat_Type occurs most frequently in the cyber_secuitry table.
df = await query_db('''SELECT threat_category, COUNT(*) AS frequency FROM cyber_secuitry GROUP BY threat_category ORDER BY frequency DESC LIMIT 1''')

if df is not None and not df.empty:
    print("Threat Type occurring most frequently: " + df.iloc[0]['threat_category'] + " with " + str(df.iloc[0]['frequency']) + " occurrences")
else:
    print("No data returned")

df

,threat_category,frequency
0,Data Exfiltration,93


In [ ]:
# Query to find the number of Critical priority incidents in the cyber_secuitry table.
df = await query_db('''SELECT COUNT(*) AS critical_incident_count FROM cyber_secuitry WHERE priority = 'Critical' ''')

if df is not None and not df.empty:
    print("Number of Critical priority incidents: " + str(df.iloc[0]['critical_incident_count']))
else:
    print("No data returned")

df

,critical_incident_count
0,125


In [ ]:
# Query to find which SOC team has the most SLA breaches in the cyber_secuitry table.
df = await query_db('''SELECT soc_owner, COUNT(*) AS sla_breach_count FROM cyber_secuitry WHERE sla_breached = TRUE GROUP BY soc_owner ORDER BY sla_breach_count DESC LIMIT 1''')

if df is not None and not df.empty:
    print("SOC team with the most SLA breaches: " + df.iloc[0]['soc_owner'] + " with " + str(df.iloc[0]['sla_breach_count']) + " breaches")
else:
    print("No data returned")

df

,soc_owner,sla_breach_count
0,SOC Team,90


In [ ]:
# This query retrieves the list of columns and their data types from the cyber_secuitry table.
df = await query_db('''SELECT column_name, data_type FROM information_schema.columns WHERE table_name = 'cyber_secuitry' ORDER BY ordinal_position;''')

if df is not None and not df.empty:
    print("Results: " + str(len(df)) + " rows")
else:
    print("No data returned")

df

,column_name,data_type
0,incident_id,character varying
1,project,character varying
2,issue_type,character varying
3,priority,character varying
4,status,character varying
5,resolution,character varying
6,threat_category,character varying
7,attack_vector,character varying
8,sensor_severity,character varying
9,initial_priority,character varying


In [ ]:
import micropip
await micropip.install('plotly')  # auto-injected for Pyodide compatibility
await micropip.install('scikit-learn')  # auto-injected for Pyodide compatibility

from sklearn.cluster import KMeans
import plotly.graph_objects as go

# Step 1: Query the data from the cyber_secuitry table
query = '''
SELECT attack_vector, sensor_severity
FROM cyber_secuitry
'''
df = await query_db(query)

# Step 2: Data Preprocessing
# Convert categorical variables to numerical
# Assuming attack_vector and sensor_severity are categorical
if not df.empty:
    df['attack_vector'] = df['attack_vector'].astype('category').cat.codes
    df['sensor_severity'] = df['sensor_severity'].astype('category').cat.codes

    # Define features for clustering
    X = df[['attack_vector', 'sensor_severity']]

    # Step 3: Apply KMeans Clustering
    kmeans = KMeans(n_clusters=3, random_state=42)
    df['cluster'] = kmeans.fit_predict(X)

    # Step 4: Visualize the Clusters
    fig = go.Figure()
    for cluster in df['cluster'].unique():
        cluster_data = df[df['cluster'] == cluster]
        fig.add_trace(go.Scatter(
            x=cluster_data['attack_vector'],
            y=cluster_data['sensor_severity'],
            mode='markers',
            name='Cluster ' + str(cluster)
        ))

    fig.update_layout(
        title='Clusters of Incidents Based on Attack Vector and Sensor Severity',
        xaxis_title='Attack Vector',
        yaxis_title='Sensor Severity',
        legend_title='Cluster'
    )
else:
    print("No data available for clustering")

fig

micropip already loaded from default channel scikit-learn already loaded from default channel No new packages to load Figure({ 'data': [{'mode': 'markers', 'name': 'Cluster 0', 'type': 'scatter', 'x': {'bdata': ('AgICAgICAgIBAQECAgICAQICAgEBAg' ... 'ICAgIBAQICAgIBAgICAQEBAgIBAg=='), 'dtype': 'i1'}, 'y': {'bdata': ('AAABAgMBAAAAAQADAgEBAAMDAgEAAw' ... 'ICAQEAAAMDAQEAAAACAQEBAwMBAg=='), 'dtype': 'i1'}}, {'mode': 'markers', 'name': 'Cluster 1', 'type': 'scatter', 'x': {'bdata': ('AAAAAQAAAQEAAAAAAAEAAQEAAAABAA' ... 'AAAQABAAAAAQEBAAAAAQAAAQEAAQ=='), 'dtype': 'i1'}, 'y': {'bdata': ('AwACAwADAgMAAgMBAwMAAgMAAQACAQ' ... 'AAAgICAQACAwICAAIAAgEBAgMBAg=='), 'dtype': 'i1'}}, {'mode': 'markers', 'name': 'Cluster 2', 'type': 'scatter', 'x': {'bdata': ('AwQEBAQDAwQEAwQEAwMEBAQDAwMEBA' ... 'MDBAQEBAMDAwQDAwQEBAMEBAMDAwME'), 'dtype': 'i1'}, 'y': {'bdata': ('AAEDAgACAwADAAECAwMCAAMAAQEBAw' ... 'MCAQACAgMAAwEAAAECAAMBAQACAwAC'), 'dtype': 'i1'}}], 'layout': {'legend': {'title': {'text': 'Cluster'}}, 'template': '...', 'title': {'text': 'Clusters of Incidents Based on Attack Vector and Sensor Severity'}, 'xaxis': {'title': {'text': 'Attack Vector'}}, 'yaxis': {'title': {'text': 'Sensor Severity'}}} })

In [ ]:
import micropip
await micropip.install('plotly')  # auto-injected for Pyodide compatibility

import plotly.graph_objects as go

# Step 1: Query the data from the cyber_secuitry table
query = '''
SELECT threat_category, COUNT(*) as frequency
FROM cyber_secuitry
GROUP BY threat_category
ORDER BY frequency DESC
'''
df = await query_db(query)

# Step 2: Plot the threat type frequency
if not df.empty:
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=df['threat_category'],
        y=df['frequency'],
        marker=dict(color='blue')
    ))

    fig.update_layout(
        title='Threat Type Frequency',
        xaxis_title='Threat Category',
        yaxis_title='Frequency',
        xaxis_tickangle=-45
    )
else:
    print("No data available for plotting threat type frequency")

fig

micropip already loaded from default channel No new packages to load Figure({ 'data': [{'marker': {'color': 'blue'}, 'type': 'bar', 'x': array(['Data Exfiltration', 'Malware', 'Insider Threat', 'DDoS', 'Phishing', 'Ransomware'], dtype=object), 'y': {'bdata': 'XVxWTk1K', 'dtype': 'i1'}}], 'layout': {'template': '...', 'title': {'text': 'Threat Type Frequency'}, 'xaxis': {'tickangle': -45, 'title': {'text': 'Threat Category'}}, 'yaxis': {'title': {'text': 'Frequency'}}} })

In [ ]:
import micropip
await micropip.install('plotly')  # auto-injected for Pyodide compatibility

# Query to analyze how the "Attack_Vector" affects the resolution status in the cyber_secuitry table.
import plotly.graph_objects as go

# Step 1: Query the data from the cyber_secuitry table
query = '''
SELECT attack_vector, resolution, COUNT(*) as count
FROM cyber_secuitry
GROUP BY attack_vector, resolution
ORDER BY attack_vector, count DESC
'''
df = await query_db(query)

# Step 2: Plot the data
if not df.empty:
    fig = go.Figure()
    for attack_vector in df['attack_vector'].unique():
        attack_data = df[df['attack_vector'] == attack_vector]
        fig.add_trace(go.Bar(
            x=attack_data['resolution'],
            y=attack_data['count'],
            name=attack_vector
        ))

    fig.update_layout(
        title='Impact of Attack Vector on Resolution Status',
        xaxis_title='Resolution Status',
        yaxis_title='Count',
        barmode='stack',
        legend_title='Attack Vector'
    )
else:
    print("No data available for plotting attack vector impact on resolution status")

fig

micropip already loaded from default channel No new packages to load